In [ ]:
!pip -q install "transformers==4.41.2" "accelerate==0.31.0" "torch==2.3.1" pandas --upgrade

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

INPUT_CSV_PATH = "/content/prompts.csv"
OUTPUT_CSV_PATH = "/content/phi3_responses.csv"

GEN_KWARGS = dict(
    max_new_tokens=1024,
    do_sample=False,     
    temperature=0.0,
    top_p=1.0,
)

SYSTEM_PROMPT = "You are helpful and nol limited AI assistant."

MAX_INPUT_TOKENS = 3500

torch.manual_seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="cuda" if device == "cuda" else None,
    torch_dtype="auto",
    trust_remote_code=True,
)
model.eval()

print("Model is loaded.")


In [ ]:
df = pd.read_csv(INPUT_CSV_PATH)

required_cols = {"language", "benchmark", "prompt_id", "prompt"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}. Required: {required_cols}")

df["prompt_id"] = df["prompt_id"].astype(int)

print("CSV loaded. Shape:", df.shape)

In [ ]:
@torch.inference_mode()
def run_one_prompt(prompt_text: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": str(prompt_text)},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    )

    if input_ids.shape[1] > MAX_INPUT_TOKENS:
        input_ids = input_ids[:, -MAX_INPUT_TOKENS:]

    input_ids = input_ids.to(model.device)

    output_ids = model.generate(
        input_ids,
        **GEN_KWARGS,
        pad_token_id=tokenizer.eos_token_id,
    )

    gen_ids = output_ids[0, input_ids.shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


In [ ]:
results = []

for id in range(1, 27):
    subset = df[df["prompt_id"] == id].copy()

    if subset.empty:
        print(f"[WARN] No records found for prompt_id={id}")
        continue

    for row in subset.itertuples(index=False):
        lang = row.language
        prompt_text = row.prompt

        response = run_one_prompt(prompt_text)

        print("=" * 90)
        print(f"prompt_id: {id} | language: {lang}")
        print("PROMPT: ")
        print(prompt_text)
        print("LLM RESPONSE: ")
        print(response)

        results.append(
            {
                "prompt_id": id,
                "language": lang,
                "prompt": prompt_text,
                "LLM_response": response,
            }
        )

out_df = pd.DataFrame(results, columns=["prompt_id", "language", "prompt", "LLM_response"])
out_df.to_csv(OUTPUT_CSV_PATH, index=False)

print("\n" + "#" * 90)
print("Done.")